# Momentum long/short backtester

Daily **time-series momentum** strategy on one asset: go long (+1) when the sum of the log returns over the last `window` days is positive, short (−1) otherwise.

- **No look-ahead**: the position is shifted by one day before it is applied.
- **Costs**: every change of position pays a proportional `fee`.
- **Data**: Yahoo Finance (`yfinance`), EUR/USD from August 2020.
- **Metric**: annualised Sharpe ratio of the net daily log returns, $\sqrt{252}\,\bar r / \sigma_r$.
- **Overfitting control**: the window (1 to 50 days) is chosen on the first 70 % of the sample (train) and evaluated on the last 30 % (test).

**Result (run of April 2026)**: the best train window (14 days, Sharpe 0.52) gives an out-of-sample Sharpe of −0.65. The in-sample optimum does not survive out of sample, which is the typical outcome of optimising a simple rule on a liquid FX pair, and the reason for the train/test split.

The last cell is a self-contained **Streamlit app** (asset picker, dates, window and fee sliders, train/test equity curves, train-only optimisation). It does not run inside Jupyter: copy it into a `.py` file and run `streamlit run <file>.py`.

In [12]:
import pandas as pd
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import streamlit as st


df=yf.download('EURUSD=X', start='2020-08-01')
# Assumptions:
# - buying and selling costs are the same
# - slippage is not modelled (not realistic)

[*********************100%***********************]  1 of 1 completed


In [26]:
def strategy(df, window=1,fee=0.0001):
    df=df.copy()  # work on a copy so the original DataFrame is not modified
    df['ret']=np.log(df.Close.pct_change() + 1) # log returns: pct_change() gives the simple return

    df['prior_n']=df.ret.rolling(window).sum() # sum of the past returns (rolling window)

    df['position']=[1 if i>0 else -1 for i in df.prior_n]  # trading signal: +1 if the momentum is positive (buy), -1 otherwise (sell / short)
    df['trade'] = df['position'].diff().abs() # change of position (detects the trades)
    df['costs'] = df['trade'] * fee # the fee is paid at each change of position (long <-> short)
    df['strat']=df.position.shift(1)*df['ret'] # gross strategy return, position shifted by one period (no trading in the past)

    df['strat_net'] = df['strat'] - df['costs'] # net of costs
    df['equity_ret'] = df['strat_net'].cumsum() # cumulative performance
    df.dropna(inplace=True)  # drop the rows with NaN (at the start, because of pct_change and rolling)

    return df


In [27]:

strategy(df, 10)


Price,Close,High,Low,Open,Volume,ret,prior_n,position,trade,costs,strat,strat_net,equity_ret
Ticker,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,,,,,,,,
Date,,,,,,,,,,,,,
2020-08-17,1.185255,1.188072,1.183040,1.185045,0,0.003042,0.005944,1,2.0,0.0002,-0.003042,-0.003242,-0.006144
2020-08-18,1.187648,1.196516,1.187536,1.187931,0,0.002017,0.009574,1,0.0,0.0000,0.002017,0.002017,-0.004127
2020-08-19,1.194786,1.195457,1.189343,1.194729,0,0.005992,0.011145,1,0.0,0.0000,0.005992,0.005992,0.001865
2020-08-20,1.184350,1.186845,1.180652,1.184175,0,-0.008773,-0.002472,-1,2.0,0.0002,-0.008773,-0.008973,-0.007108
2020-08-21,1.187366,1.188637,1.175490,1.187085,0,0.002544,-0.000237,-1,0.0,0.0000,-0.002544,-0.002544,-0.009652
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-30,1.149201,1.152339,1.144387,1.149188,0,-0.003742,0.004964,1,0.0,0.0000,-0.003742,-0.003742,-0.152685
2026-03-31,1.146000,1.153882,1.144925,1.145790,0,-0.002789,-0.003444,-1,2.0,0.0002,-0.002789,-0.002989,-0.155674


In [18]:
df2=strategy(df)

def sharpe_ratio(df2,rf=0.0):
    r = df2['strat_net'].dropna()
    if r.std() == 0:
        return 0
    return np.sqrt(252) * (r.mean()-rf) / r.std() # annualised Sharpe ratio

sharpe_ratio(df2,0.0)

np.float64(-0.11884290166403155)

In [20]:
def optimize_window(df2, windows=range(1, 51),fee=0.0001):
    results = []
    for w in windows:
        df_temp = strategy(df2, window=w,fee=fee) # apply the strategy on an independent copy
        results.append((w, sharpe_ratio(df_temp))) # Sharpe ratio, stored with its window
    return results
# split the data to avoid overfitting
split = int(len(df2) * 0.7)
df_train = df2.iloc[:split]
df_test = df2.iloc[split:]
# optimisation on the train data only
results = optimize_window(df_train, windows=range(1, 51),fee=0.0001)
best_window = max(results, key=lambda x: x[1])[0]
best_sharpe = max(results, key=lambda x: x[1])[1]

print("Best window (train):", best_window)
print("Best Sharpe (train):", best_sharpe)

# evaluation on the TEST data (out of sample)
df_test_strat = strategy(df_test, window=best_window)

test_sharpe = sharpe_ratio(df_test_strat)

print("Out-of-sample Sharpe:", test_sharpe)

Best window (train): 14
Best Sharpe (train): 0.5205448964919772
Out-of-sample Sharpe: -0.6462642051049067


In [ ]:
import streamlit as st
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def strategy(df, window=1,fee=0.0001):
    df=df.copy()  # work on a copy so the original DataFrame is not modified
    df['ret']=np.log(df.Close.pct_change() + 1) # log returns: pct_change() gives the simple return
    
    df['prior_n']=df.ret.rolling(window).sum() # sum of the past returns (rolling window)
   
    df['position']=[1 if i>0 else -1 for i in df.prior_n]  # trading signal: +1 if the momentum is positive (buy), -1 otherwise (sell / short)
    df['trade'] = df['position'].diff().abs() # change of position (detects the trades)
    df['costs'] = df['trade'] * fee # the fee is paid at each change of position (long <-> short)
    df['strat']=df.position.shift(1)*df['ret'] # gross strategy return, position shifted by one period (no trading in the past)
   
    df['strat_net'] = df['strat'] - df['costs'] # net of costs
    df['equity_ret'] = df['strat_net'].cumsum()# cumulative performance
    df.dropna(inplace=True)  # drop the rows with NaN (at the start, because of pct_change and rolling)
   
    return df

def optimize_window(df2, windows=range(1, 51),fee=0.0001):
    results = []
    for w in windows:
        df_temp = strategy(df2, window=w,fee=fee) # apply the strategy on an independent copy
        results.append((w, sharpe_ratio(df_temp))) # Sharpe ratio, stored with its window
    return results

def sharpe_ratio(df2,rf=0.0):
    r = df2['strat_net'].dropna() 
    if r.std() == 0:
        return 0
    return np.sqrt(252) * (r.mean()-rf) / r.std() # annualised Sharpe ratio of the returns net of costs
# Dashboard
# Asset selection
TICKERS = {
    "EUR/USD"        : "EURUSD=X",
    "BTC/USD"        : "BTC-USD",
    "S&P 500"        : "^GSPC",
    "Gold"           : "GC=F",
    "Apple"          : "AAPL",
    "Nasdaq"         : "^IXIC",
}

st.title("📈 Quant Strategy Dashboard")

ticker_label = st.selectbox("Index / asset", options=list(TICKERS.keys()))
ticker       = TICKERS[ticker_label]

# Period selection

col_d1, col_d2 = st.columns(2)
start_date = col_d1.date_input("Start date", value=pd.Timestamp("2020-08-01"))
end_date   = col_d2.date_input("End date",     value=pd.Timestamp("today"))

if start_date >= end_date:
    st.error("The start date must be before the end date.")
    st.stop()

# Data loading (cached so that it is not downloaded again at each interaction)

@st.cache_data
def load_data(ticker, start, end):
    df = yf.download(ticker, start=start_date, end=end_date)
    if df.empty:
        st.error(f"Could not download the data for {ticker}.")
        st.stop()
    return df

df = load_data(ticker, start=start_date, end=end_date)

# Train/test split, once, outside the functions

split = int(len(df) * 0.7)
df_train = df.iloc[:split]
df_test  = df.iloc[split:]


# Dates shown so that the user knows which period each set covers
col1, col2 = st.columns(2)
col1.caption(f"Train: {df_train.index[0].date()} → {df_train.index[-1].date()}")
col2.caption(f"Test:  {df_test.index[0].date()}  → {df_test.index[-1].date()}")


# User parameters

st.subheader("Parameters")
window = st.slider("Window (momentum)", 1, 50, 10)
fee = st.slider("Transaction cost (per trade)", min_value=0.0, max_value=0.001, value=0.0001, step=0.00001,format="%.5f")


# Apply the strategy on the full dataset, then split it
# (avoids a flat start of the test period caused by a "cold" rolling window)

df_full_strat  = strategy(df, window=window, fee=fee)

split_idx      = df_full_strat.index.get_loc(df_train.index[-1])
df_train_strat = df_full_strat.iloc[:split_idx+1]
df_test_strat  = df_full_strat.iloc[split_idx+1:]

# Sharpe ratios, train and test side by side
# (the gap between them reveals overfitting)

sr_train = sharpe_ratio(df_train_strat)
sr_test  = sharpe_ratio(df_test_strat)

st.subheader("Performance")
m1, m2 = st.columns(2)
m1.metric("Sharpe — Train",        round(sr_train, 3))
m2.metric("Sharpe — Out-of-sample", round(sr_test,  3),
          delta=round(sr_test - sr_train, 3))   # the delta shows the degradation, if any


# Equity curves, train AND test on the same chart
# The two periods are concatenated for a continuous display

st.subheader("Equity curve")

def equity_curve(df_strat):
    return np.exp(df_strat[['ret', 'strat_net']].cumsum())

eq_train = equity_curve(df_train_strat)
eq_test  = equity_curve(df_test_strat)

# Shift the test curve so that it starts from the last train value (continuous curve)
offset = eq_train.iloc[-1] - eq_test.iloc[0]
eq_test_adj = eq_test + offset

equity_full = pd.concat([eq_train, eq_test_adj])
equity_full.columns = ['Buy & Hold', 'Strategy']

fig, ax = plt.subplots(figsize=(10, 4))
equity_full['Buy & Hold'].plot(ax=ax, label='Buy & Hold', color='steelblue', linewidth=1)
equity_full['Strategy'].plot(ax=ax,   label='Strategy',   color='darkorange', linewidth=1)

# Vertical line at the train/test split
ax.axvline(df_train_strat.index[-1], color='gray', linestyle='--', linewidth=0.8, label='Train / Test')
ax.legend(fontsize=9)
ax.set_title("Equity curve — train + out-of-sample", fontsize=11)
ax.set_ylabel("Portfolio value (base 1)")
st.pyplot(fig)
plt.close(fig)   # frees the matplotlib memory


# Optimisation: on the train set only, never on the full df

st.subheader("Window optimisation")

if st.button("Run the optimisation (train set only)"):
    with st.spinner("Computing..."):
        results = optimize_window(df_train, windows=range(1, 51), fee=fee)

    best_window = max(results, key=lambda x: x[1])[0]
    best_sharpe = max(results, key=lambda x: x[1])[1]

    # Out-of-sample check with the best window found
    df_best_test = strategy(df_test, window=best_window, fee=fee)
    oos_sharpe   = sharpe_ratio(df_best_test)

    b1, b2, b3 = st.columns(3)
    b1.metric("Best window",          best_window)
    b2.metric("Best train Sharpe", round(best_sharpe, 3))
    b3.metric("Out-of-sample Sharpe",   round(oos_sharpe,  3),
              delta=round(oos_sharpe - best_sharpe, 3))

    # Chart: Sharpe ratio against the window
    windows_list = [w for w, s in results]
    sharpes_list = [s for w, s in results]

    fig2, ax2 = plt.subplots(figsize=(8, 3))
    ax2.plot(windows_list, sharpes_list, color='steelblue', linewidth=1.2)
    ax2.axvline(best_window, color='darkorange', linestyle='--', linewidth=1, label=f'Best = {best_window}')
    ax2.axhline(0, color='gray', linewidth=0.5)
    ax2.set_xlabel("Window")
    ax2.set_ylabel("Sharpe ratio")
    ax2.set_title("Sharpe ratio vs momentum window (train)", fontsize=11)
    ax2.legend(fontsize=9)
    st.pyplot(fig2)
    plt.close(fig2)